# 3. Evaluate saved TinyStories checkpoints (no training)

Enable **GPU** and **Internet**. Attach (1) the private prepared TinyStories Dataset from notebook 1 and (2) the saved **Output of notebook 2** using Add Input → Notebook Output, or a Dataset made from that output.

An interrupted session can be evaluated if its checkpoint files were preserved in the saved output. This notebook cannot recover files absent from that output. It accepts one model (AR or NCM) and partial training; it never starts training or downloads TinyStories. GPT-2-large is downloaded on Kaggle for quality scoring.

Change the input paths below. Available checkpoints and their actual update counts are printed before evaluation. Partial training and unequal budgets are explicitly labelled, not treated as a paper reproduction. Results are saved under `/kaggle/working/ncm-evaluation/`.


In [ ]:
import os
from pathlib import Path
import subprocess
import sys

REPO_URL = 'https://github.com/kuchidareo/3rd-contribution.git'  # Set the 3rd-contribution repository URL.
REPO_REF = 'main'  # Prefer a full commit containing the baseline implementation.
CHECKOUT = Path('/kaggle/working/3rd-contribution')
REPO = CHECKOUT / '261119-NCM-replicating/ncm'

def run(command, cwd=REPO):
    print('+', ' '.join(map(str, command)), flush=True)
    subprocess.run(list(map(str, command)), cwd=cwd, check=True)

if not CHECKOUT.exists():
    run(['git', 'clone', REPO_URL, CHECKOUT], cwd='/kaggle/working')
run(['git', 'fetch', 'origin', REPO_REF], cwd=CHECKOUT)
run(['git', 'checkout', '--detach', 'FETCH_HEAD'], cwd=CHECKOUT)
required = [REPO / 'baseline_data/prepare.py', REPO / 'training/train_ar.py',
            REPO / 'config/baseline/tinystories.yaml', REPO / 'training/profiles.py']
if any(not p.is_file() for p in required):
    raise RuntimeError('This revision lacks the baseline code. Commit/push the local changes, then set REPO_REF.')
os.chdir(REPO)
# Keep all downloads and environment records inside the preparation/training output.
run(['bash', 'scripts/kaggle_setup.sh'])


In [ ]:
import json
import torch
from omegaconf import OmegaConf

DATASET_ROOT = Path('/kaggle/input/ncm-tinystories/ncm-tinystories')
# EDIT: attached saved notebook output (may include ncm-experiment/paper-fp16/).
TRAINING_OUTPUT = Path('/kaggle/input/previous-training-output')
MODELS = ['ar', 'ncm']  # Missing models are reported and skipped.
CHECKPOINT = 'best'  # 'latest' also supported. Falls back to latest if best is absent.
# Optional exact paths, useful if TRAINING_OUTPUT contains multiple experiments.
CHECKPOINT_PATHS = {'ar': None, 'ncm': None}
RESULT_TAG = 'saved-checkpoints-01'  # Change for a new measurement; existing results can be reused.
RESULTS = Path('/kaggle/working/ncm-evaluation') / RESULT_TAG
# None preserves the saved protocol. Lower values are recorded evaluation deviations.
GENERATION_BATCH_SIZE = None  # e.g. 4 if saved batch 128 exceeds GPU memory.
SCORING_BATCH_SIZE = 8  # Scorer batching only; keeps GPT-2-large and the saved sample protocol.
EVAL_PRECISION = None  # None preserves saved precision; use 'fp16' for BF16 checkpoints on T4.

if not torch.cuda.is_available():
    raise RuntimeError('Enable GPU for evaluation.')
if not (DATASET_ROOT / 'artifacts/tinystories/tokenizer.pt').is_file():
    raise FileNotFoundError('Set DATASET_ROOT to the prepared Dataset containing artifacts/tinystories/.')
if CHECKPOINT not in {'best', 'latest'}:
    raise ValueError('CHECKPOINT must be best or latest.')
if not MODELS or len(set(MODELS)) != len(MODELS) or set(MODELS) - {'ar', 'ncm'}:
    raise ValueError('MODELS must contain ar and/or ncm without duplicates.')
RESULTS.mkdir(parents=True, exist_ok=True)


In [ ]:
# Read metadata on CPU; do not load optimizer state onto the GPU.
selected = {}
summary = []
for kind in MODELS:
    explicit = CHECKPOINT_PATHS.get(kind)
    if explicit:
        candidates = [Path(explicit)]
    else:
        candidates = sorted(TRAINING_OUTPUT.glob(f'**/runs/{kind}/{CHECKPOINT}.pt'))
        if not candidates and CHECKPOINT == 'best':
            candidates = sorted(TRAINING_OUTPUT.glob(f'**/runs/{kind}/latest.pt'))
            if candidates:
                print(f'{kind}: best.pt absent; using latest.pt.')
    if not candidates:
        print(f'{kind}: no saved checkpoint found; skipping. Check attached output if unexpected.')
        continue
    if len(candidates) != 1:
        raise ValueError(f'{kind}: multiple experiments found. Set CHECKPOINT_PATHS[{kind!r}] explicitly: {candidates}')
    checkpoint = candidates[0]
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    header = torch.load(checkpoint, map_location='cpu', weights_only=False, mmap=True)
    if header.get('kind') != kind or 'config' not in header.get('metadata', {}):
        raise ValueError(f'{checkpoint}: requires a training checkpoint with matching kind and config metadata.')
    cfg = OmegaConf.create(header['metadata']['config'])
    step = int(header['step'])
    target = int(cfg.training.generator.steps)
    # Relocate paths from the old session to this attached Dataset. Architecture and
    # generation/scoring settings come from the checkpoint, not today's profile defaults.
    cfg.paths.artifacts = str(DATASET_ROOT / 'artifacts/tinystories')
    cfg.paths.tokenizer_checkpoint = str(DATASET_ROOT / 'artifacts/tinystories/tokenizer.pt')
    cfg.paths.data = str(DATASET_ROOT / 'prepared')
    cfg.paths.codes = str(DATASET_ROOT / 'codes')
    cfg.device = 'cuda'
    if GENERATION_BATCH_SIZE is not None:
        cfg.evaluation.batch_size = GENERATION_BATCH_SIZE
    if EVAL_PRECISION is not None:
        cfg.precision = EVAL_PRECISION
    if cfg.precision == 'bf16' and not (torch.cuda.get_device_capability()[0] >= 8 and torch.cuda.is_bf16_supported()):
        raise RuntimeError('Saved precision is BF16; select a native BF16 GPU or explicitly set EVAL_PRECISION="fp16" (evaluation deviation).')
    config_path = RESULTS / f'{kind}-evaluation.yaml'
    if config_path.exists() and OmegaConf.load(config_path) != cfg:
        raise ValueError('Evaluation settings changed. Use a new RESULT_TAG.')
    OmegaConf.save(cfg, config_path)
    selected[kind] = dict(checkpoint=checkpoint, config=config_path)
    row = dict(model=kind, checkpoint=str(checkpoint), update=step, planned_updates=target,
               partial_training=step < target, stop_reason=header.get('stop_reason'),
               text_positions_seen=header.get('text_positions_seen'))
    summary.append(row)
    print(json.dumps(row, indent=2))
    del header, cfg
if not selected:
    raise FileNotFoundError('No checkpoints available. Attach the saved training output and check TRAINING_OUTPUT. You can list paths with list(Path("/kaggle/input").glob("**/latest.pt")).')
(RESULTS / 'checkpoint-summary.json').write_text(json.dumps(summary, indent=2))
print('Partial checkpoints are valid for inspection, but do not establish a completed paper reproduction.')


In [ ]:
# Separate subprocesses release generator memory before loading GPT-2-large.
# A completed generation/scoring stage can be reused after a later stage fails.
from training.runtime import sha256
for kind, item in selected.items():
    out = RESULTS / kind
    generation = out / 'generation.json'
    if generation.exists():
        saved = json.loads(generation.read_text())
        if saved['checkpoint_sha256'] != sha256(item['checkpoint']):
            raise ValueError('Checkpoint changed. Use a new RESULT_TAG.')
        from training.runtime import load_config
        expected = OmegaConf.to_container(load_config(item['config']), resolve=True)
        if saved['config'] != expected:
            raise ValueError('Generation config changed. Use a new RESULT_TAG.')
        print(f'{kind}: reusing completed generation.')
    else:
        run([sys.executable, '-m', 'evaluation.evaluate', '--config', item['config'],
             '--model', kind, '--checkpoint', item['checkpoint'], '--out', out])
    quality = out / 'quality.json'
    if quality.exists():
        if json.loads(quality.read_text())['generation_sha256'] != sha256(generation):
            raise ValueError('Quality does not match generation. Use a new RESULT_TAG.')
        print(f'{kind}: reusing completed quality scoring.')
    else:
        run([sys.executable, '-m', 'evaluation.score', '--run', out,
             '--batch-size', str(SCORING_BATCH_SIZE)])


In [ ]:
import pandas as pd
from IPython.display import display, Image
rows = []
for item in summary:
    kind = item['model']
    generation = json.loads((RESULTS / kind / 'generation.json').read_text())
    quality = json.loads((RESULTS / kind / 'quality.json').read_text())
    timing = generation['benchmark']
    rows.append(dict(**item, generative_perplexity=quality['generative_perplexity'],
                     mean_unigram_entropy=quality['mean_unigram_entropy'],
                     latency_seconds=timing['median_batch_latency_seconds'],
                     tokens_per_second=timing['tokens_per_second'],
                     forward_passes_per_batch=timing['forward_passes_per_batch'][0],
                     peak_allocated_bytes=timing['peak_allocated_bytes'],
                     generator_parameters=generation['generator']))
    print(f'\n{kind.upper()} generated examples:')
    for line in (RESULTS / kind / 'samples.jsonl').read_text().splitlines()[:3]:
        print(json.loads(line)['text'], '\n')
table = pd.DataFrame(rows)
table.to_csv(RESULTS / 'summary.csv', index=False)
display(table)
if {'ar', 'ncm'} <= set(selected):
    # The comparison tool records all mismatches; this permits diagnostic partial runs.
    run([sys.executable, '-m', 'evaluation.compare', '--ar', RESULTS / 'ar',
         '--ncm', RESULTS / 'ncm', '--out', RESULTS / 'comparison', '--allow-unmatched'])
    display(pd.read_csv(RESULTS / 'comparison/comparison.csv'))
    display(Image(filename=str(RESULTS / 'comparison/comparison.png')))
else:
    print('Only one model is available. Its scores are saved; AR/NCM comparison requires both checkpoints.')
print('Save this notebook version with outputs:', RESULTS)


## Train NCM in the next session

In notebook 2 set `TRAIN_MODELS = ['ncm']`. Keep the intended `PROFILE` and `PRECISION`. Set `NCM_RESUME` to an existing NCM `latest.pt` if available; otherwise leave it `None` to start NCM. You do not need `AR_RESUME` when training only NCM. Attach the same prepared Dataset; no new dataset download is needed.

Save training outputs containing `runs/ncm/latest.pt`, `runs/ncm/best.pt` and the experiment config. Then attach both AR and NCM outputs here; use `CHECKPOINT_PATHS` to select checkpoints from different attached outputs. Preserve both best and latest for later resume. Evaluating best selects validation loss, not guaranteed lowest generative perplexity.
